# Data Preprocessing

Dataset: _AI-Driven Resume Screening_ (Kaggle, ~30,000 samples)


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("ai_resume_screening.csv")
df.head()

,years_experience,skills_match_score,education_level,project_count,resume_length,github_activity,shortlisted
0,6,84.7,Bachelors,7,234,158,No
1,3,59.1,Masters,5,502,77,No
2,12,100.0,Masters,12,753,381,Yes
3,14,66.8,High School,8,529,407,Yes
4,10,99.6,Bachelors,10,754,331,Yes


## Exploratory Data Analysis


In [3]:
print(f"Shape: {df.shape}")
df.info()

Shape: (30000, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   years_experience    30000 non-null  int64  
 1   skills_match_score  30000 non-null  float64
 2   education_level     30000 non-null  object 
 3   project_count       30000 non-null  int64  
 4   resume_length       30000 non-null  int64  
 5   github_activity     30000 non-null  int64  
 6   shortlisted         30000 non-null  object 
dtypes: float64(1), int64(4), object(2)
memory usage: 1.6+ MB


In [4]:
df.describe()

,years_experience,skills_match_score,project_count,resume_length,github_activity
count,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000
mean,7.506567,73.682653,10.646267,572.584700,325.260667
std,4.624104,16.765909,4.634047,178.709918,159.951803
min,0.000000,0.500000,0.000000,150.000000,0.000000
25%,3.750000,62.100000,7.000000,441.000000,202.000000
50%,7.000000,74.300000,10.000000,574.000000,321.000000
75%,12.000000,86.500000,14.000000,709.000000,443.000000
max,15.000000,100.000000,25.000000,900.000000,842.000000


In [5]:
df["shortlisted"].value_counts()
# Class distribution — imbalanced (~70% shortlisted)

shortlisted
Yes    20966
No      9034
Name: count, dtype: int64

In [8]:
print("Missing values:")
print(df.isnull().sum())
print(f"\nDuplicate rows: {df.duplicated().sum()}")

Missing values:
years_experience      0
skills_match_score    0
education_level       0
project_count         0
resume_length         0
github_activity       0
shortlisted           0
dtype: int64

Duplicate rows: 0


## Feature Selection

`github_activity` is excluded as it requires a separate data source (GitHub API) and cannot be extracted from a resume PDF.


In [9]:
df = df.drop(columns=["github_activity"])

X = df.drop("shortlisted", axis=1)
y = df["shortlisted"].map({"Yes": 1, "No": 0})

print(f"Features ({X.shape[1]}): {X.columns.tolist()}")
print(f"Samples:  {X.shape[0]}")
print(f"Target distribution:\n{y.value_counts()}")

Features (5): ['years_experience', 'skills_match_score', 'education_level', 'project_count', 'resume_length']
Samples:  30000
Target distribution:
shortlisted
1    20966
0     9034
Name: count, dtype: int64


## Train / Validation / Test Split


In [10]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print(f"Train:      {X_train.shape[0]:,} samples (70%)")
print(f"Validation: {X_val.shape[0]:,} samples (15%)")
print(f"Test:       {X_test.shape[0]:,} samples (15%)")

Train:      21,000 samples (70%)
Validation: 4,500 samples (15%)
Test:       4,500 samples (15%)


## Encoding

`education_level` is ordinally encoded (High School < Bachelors < Masters < PhD) since it has a natural order.


In [11]:
education_order = {"High School": 0, "Bachelors": 1, "Masters": 2, "PhD": 3}

X_train_enc = X_train.copy()
X_val_enc   = X_val.copy()
X_test_enc  = X_test.copy()

for split in [X_train_enc, X_val_enc, X_test_enc]:
    split["education_level"] = split["education_level"].map(education_order)

X_train_enc.head()

,years_experience,skills_match_score,education_level,project_count,resume_length
13163,1,48.1,1,4,740
29921,11,93.7,3,13,836
2369,5,77.3,1,10,592
5257,12,96.0,3,19,603
3640,1,71.6,2,4,509


## Class Weights

Computed from the training set to handle the 70/30 imbalance.


In [12]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.array([0, 1])
cw = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, cw))

print(f"  Not Shortlisted (0): {cw[0]:.4f}")
print(f"  Shortlisted     (1): {cw[1]:.4f}")

  Not Shortlisted (0): 1.6603
  Shortlisted     (1): 0.7155


## Feature Scaling


In [13]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_enc)
X_val_scaled   = scaler.transform(X_val_enc)
X_test_scaled  = scaler.transform(X_test_enc)

print("Preprocessing complete.")
print(f"Train: {X_train_scaled.shape} | Val: {X_val_scaled.shape} | Test: {X_test_scaled.shape}")

Preprocessing complete.
Train: (21000, 5) | Val: (4500, 5) | Test: (4500, 5)
